In [2]:
# pip install langgraph

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator

# --- 0. 각 에이전트의 역할을 정의 (함수 또는 클래스로) ---
# 이전 코드의 model_1, model_2, model_3 함수를 그대로 사용하거나
# 각 모델을 호출하는 Agent 클래스를 만들 수 있습니다.
# 여기서는 간단히 함수를 그대로 사용하겠습니다.
from your_models import model_1_image_to_empty_room, model_2_text_to_prompt, model_3_image_to_image

# --- 1. 워크플로우의 상태(State) 정의 ---
# 각 단계의 결과물을 저장하고 공유하는 데이터 구조
class AgentState(TypedDict):
    original_image: str
    user_requirements: str
    empty_room_image: str
    generated_prompt: str
    final_image: str
    # 어떤 병렬 작업이 완료되었는지 추적
    parallel_tasks_completed: Annotated[list[str], operator.add]

# --- 2. 그래프의 노드(Node) 정의 ---
# 각 노드는 에이전트가 수행하는 작업에 해당합니다.

def run_image_refiner(state: AgentState):
    print("--- 이미지 정제 에이전트 작업 중 ---")
    result = model_1_image_to_empty_room(state["original_image"])
    return {"empty_room_image": result, "parallel_tasks_completed": ["image_refined"]}

def run_prompt_generator(state: AgentState):
    print("--- 프롬프트 생성 에이전트 작업 중 ---")
    result = model_2_text_to_prompt(state["user_requirements"])
    return {"generated_prompt": result, "parallel_tasks_completed": ["prompt_generated"]}

def run_interior_designer(state: AgentState):
    print("--- 인테리어 디자이너 에이전트 작업 중 ---")
    result = model_3_image_to_image(state["empty_room_image"], state["generated_prompt"])
    return {"final_image": result}

# --- 3. 조건부 엣지(Edge) 정의 ---
# 병렬 작업이 모두 끝났는지 확인하는 로직
def should_continue_to_designer(state: AgentState):
    if "image_refined" in state["parallel_tasks_completed"] and "prompt_generated" in state["parallel_tasks_completed"]:
        return "run_designer"  # 둘 다 끝났으면 디자이너 노드로
    else:
        return END # 아직 안 끝났으면 대기 (실제로는 더 복잡한 대기 로직 필요)

# --- 4. 그래프(Graph) 생성 및 연결 ---
workflow = StateGraph(AgentState)

# 병렬 실행을 위한 진입점(entry point) 추가
workflow.add_node("image_refiner", run_image_refiner)
workflow.add_node("prompt_generator", run_prompt_generator)
workflow.add_node("designer", run_interior_designer)

# 시작 지점 설정
workflow.set_entry_point("image_refiner")
workflow.set_entry_point("prompt_generator")

# 노드 연결
workflow.add_edge("image_refiner", "designer") # 실제로는 조건부 엣지로 가야함
workflow.add_edge("prompt_generator", "designer") # 이 부분은 개념적 표현
# LangGraph 최신 버전에서는 조건부 엣지를 통해 병렬 작업의 'join'을 더 정교하게 처리합니다.
# workflow.add_conditional_edges(...)

workflow.add_edge("designer", END)

# 그래프 컴파일
app = workflow.compile()

# --- 5. 실행 ---
inputs = {
    "original_image": "내_방_사진.jpg",
    "user_requirements": "모던하고 미니멀한 스타일로 바꿔줘",
    "parallel_tasks_completed": []
}
# stream()을 사용하면 각 단계의 진행 상황을 실시간으로 볼 수 있습니다.
for event in app.stream(inputs):
    print(event)

--- 이미지 정제 에이전트 작업 중 ---
🚀 모델 1 시작: 원본 이미지를 빈 방으로 변환 중...
--- 프롬프트 생성 에이전트 작업 중 ---
🚀 모델 2 시작: 요구사항을 프롬프트로 변환 중...
✅ 모델 2 완료!
{'prompt_generator': {'generated_prompt': "'모던하고 미니멀한 스타일로 바꿔줘'를 반영한 프롬프트: 'A modern, minimalist living room with oak wood floors and a large window.'", 'parallel_tasks_completed': ['prompt_generated']}}
✅ 모델 1 완료!
{'image_refiner': {'empty_room_image': "'내_방_사진.jpg'의_빈_방_이미지.jpg", 'parallel_tasks_completed': ['image_refined']}}
--- 인테리어 디자이너 에이전트 작업 중 ---

🚀 모델 3 시작: 빈 방에 프롬프트를 적용하여 최종 이미지 생성 중...
   - 입력 이미지: '내_방_사진.jpg'의_빈_방_이미지.jpg
   - 적용 프롬프트: '모던하고 미니멀한 스타일로 바꿔줘'를 반영한 프롬프트: 'A modern, minimalist living room with oak wood floors and a large window.'
🎉 모델 3 완료!
{'designer': {'final_image': '최종_인테리어_결과물.jpg'}}
